---
title: "DRG Cleaning"

author: "Carlos Resurreccion"

date: "2024-07-01"

---

In [187]:
knitr::opts_chunk$set(echo = TRUE)


## Load Required Libraries

In [188]:
options(verbose = FALSE)
options(warn = -1)


In [189]:
library(here)
# source(here("data-cleaning", "r_scripts", "libraries.R"))
source(here("data-cleaning", "r_scripts", "everything.R"))


In [190]:
options(warn = 1)


## Set Parameters

### Year to Load, Version, and Parameters

In [191]:
# source(here("data-cleaning", "r_scripts", "parameters.R"))


## Source Data Formats

In [192]:
# source(here("data-cleaning", "r_scripts", "data-formats.R"))
# source(here("data-cleaning", "r_scripts", "file-paths.R"))


## Source Functions

In [193]:
# source(here("data-cleaning", "r_scripts", "general-functions.R"))
# # source(here("data-cleaning", "r_scripts", "clean-data-mini-functions.R"))
# # source(here("data-cleaning", "r_scripts", "profvis.R"))
# source(here("data-cleaning", "r_scripts", "main-functions.R"))
# source(here("data-cleaning", "r_scripts", "icd-functions.R"))
# source(here("data-cleaning", "r_scripts", "rvs-functions.R"))
# source(here("data-cleaning", "r_scripts", "pdx-functions.R"))
# source(here("data-cleaning", "r_scripts", "grouper-functions.R"))


## Load Mapping Data

In [194]:
proc <- fread(here(path_to_excel, "proc.csv"))
proc[, CODE := as.character(CODE)]
# head(proc)

rvs_icd9 <- fread(here(path_to_aux, "rvs_icd9cm.csv"),
  select = c("rvs", "icd9cm")
)
rvs_icd9[, rvs := as.character(rvs)]
rvs_icd9[, icd9cm := as.character(icd9cm * 100)]
rvs_icd9 <- merge(rvs_icd9, proc[, .(CODE, DRGUSE)],
  by.x = "icd9cm", by.y = "CODE", all.x = TRUE
)
rvs_icd9[, is_drg := !is.na(DRGUSE) & DRGUSE]
rvs_icd9 <- rvs_icd9[!is.na(rvs) & !is.na(icd9cm), -"DRGUSE"]
# head(rvs_icd9)

acr_rvs <- fread(here(path_to_aux, "acr_rvs.csv"))
# head(acr_rvs)

# Read in the data.table
tdrg_icd10 <- fread(here(path_to_aux, "i10.csv"))

# Set the key if not already set
setkey(tdrg_icd10, "CODE")
# head(tdrg_icd10)

# Subset and assign the result to acc_pdx
acc_pdx <- tdrg_icd10[ACCPDX == "Y", CODE]

# Optional: if CODEs are not unique in tdrg_icd10
acc_pdx <- unique(acc_pdx)


## Read Data

### Reading Data

In [195]:
options(verbose = FALSE)
options(warn = -1)

dt <- main_read_function()

if (file.exists(total_rows_file)) {
  total_rows <- readRDS(total_rows_file)
} else {
  total_rows <- fread(full_claims, select = 1L, header = TRUE)[, .N]
  saveRDS(total_rows, file = total_rows_file)
}


[1] "Sampled file exists. Reading the sampled file..."
[1] "Sampled file matches sample size."


In [196]:
options(warn = 1)


## Data Processing

### Data Cleaning

#### Not chunking

In [197]:
if (!to_chunk) {
  if (to_profvis) {
    tic("Total execution time:")
    p <- profvis({
      dt <- clean_data(dt)
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "clean_data.html"),
      selfcontained = TRUE
    )
  } else {
    tic("Total execution time:")
    dt <- clean_data(dt)
  }
}


#### Chunking

In [198]:
if (to_chunk) {
  tic("Total execution time:")
  if (to_profvis) {
    p <- profvis({
      num_cores <- max(1, availableCores() - 1)
      chunk_size <- ceiling(nrow(dt) / num_cores)
      chunks <- split(dt, rep(1:num_cores,
        each = chunk_size,
        length.out = nrow(dt)
      ))
      # Plan for parallel processing
      plan(multisession, workers = num_cores)
      # Process each chunk in parallel
      results <- future_lapply(chunks, process_chunk,
        future.seed = global_seed
      )
      # Combine processed chunks
      processed_chunks <- lapply(results, function(res) res$chunk)
      dt <- rbindlist(processed_chunks)
      dt <- replace_empty_with_na(dt, to_view_checks)

      # Combine summaries
      summaries <- lapply(results, function(res) res$summary)
      consolidated_summary <- list(
        rename_success = all(sapply(summaries, function(s) s$rename_success)),
        compone = combine_comparison_tables(
          summaries,
          "compone",
          rows_to_show = 10
        ),
        comptwo = combine_comparison_tables(
          summaries,
          "comptwo",
          rows_to_show = 10
        ),
        unmapone = unique(unlist(lapply(summaries, function(s) s$unmapone))),
        unmaptwo = unique(unlist(lapply(summaries, function(s) s$unmaptwo))),
        unmapthree = unique(unlist(lapply(summaries, function(s) s$unmapthree))),
        unmapfour = unique(unlist(lapply(summaries, function(s) s$unmapfour)))
      )
      print(consolidated_summary) # Or save the consolidated summary as needed
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "parallelized.html"),
      selfcontained = TRUE
    )
  } else {
    num_cores <- max(1, availableCores() - 1)
    chunk_size <- ceiling(nrow(dt) / num_cores)
    chunks <- split(dt, rep(1:num_cores,
      each = chunk_size,
      length.out = nrow(dt)
    ))
    # Plan for parallel processing
    plan(multisession, workers = num_cores)
    # Process each chunk in parallel
    results <- future_lapply(chunks, process_chunk,
      future.seed = global_seed
    )
    # Combine processed chunks
    processed_chunks <- lapply(results, function(res) res$chunk)
    dt <- rbindlist(processed_chunks)
    dt <- replace_empty_with_na(dt, to_view_checks)

    # Combine summaries
    summaries <- lapply(results, function(res) res$summary)
    consolidated_summary <- list(
      rename_success = all(sapply(summaries, function(s) s$rename_success)),
      compone = combine_comparison_tables(
        summaries,
        "compone",
        rows_to_show = 10
      ),
      comptwo = combine_comparison_tables(
        summaries,
        "comptwo",
        rows_to_show = 10
      ),
      unmapone = unique(unlist(lapply(summaries, function(s) s$unmapone))),
      unmaptwo = unique(unlist(lapply(summaries, function(s) s$unmaptwo))),
      unmapthree = unique(unlist(lapply(summaries, function(s) s$unmapthree))),
      unmapfour = unique(unlist(lapply(summaries, function(s) s$unmapfour)))
    )
    print(consolidated_summary) # Or save the consolidated summary as needed
  }
}


[1] "Viewing checks"
     clin_c1_orig clin_c1
           <char>  <char>
  1:        A97.1    A971
  2:        I21.9    I219
  3:        K29.1    K291
  4:        J06.9    J069
  5:        N39.0    N390
 ---                     
691:       J18.92   J1892
692:        I63.8    I638
693:        O14.1    O141
694:        A06.0    A060
695:       J18.92   J1892
   clin_c2_orig clin_c2
         <char>  <char>
1:        E86.1    E861
2:        I63.4    I634
3:        I61.5    I615


|CODE  | Counts|
|:-----|------:|
|77401 |     20|
|90375 |     10|
|77418 |      9|
|77261 |      2|
|76942 |      1|


|CODE  | Counts|
|:-----|------:|
|77418 |      1|
|77401 |      1|


|Column     | "" Replaced| "NA" Replaced| "character(0)" Replaced|
|:----------|-----------:|-------------:|-----------------------:|
|date_ref   |        1481|             0|                       0|
|date_check |         110|             0|                       0|
|clin_acc   |          54|             0|                   

### Map codes and Find PDx (if not chunking)

#### Map Codes

In [199]:
# Map RVS codes
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
      map_then_compare_icd_mappings(
        tdrg_icd10,
        rows_to_show = 10,
        invalid_rows_to_show = 10
      )

      # Replace empty strings in character and factor columns with NA
      # dt <- replace_empty_with_na(dt, to_view_checks)
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "map_rvs.html"),
      selfcontained = TRUE
    )
  } else {
    dt[, icd9_list := map_rvs_icd9(clin_rvs, rvs_icd9)]
    map_then_compare_icd_mappings(
      tdrg_icd10,
      rows_to_show = 10,
      invalid_rows_to_show = 10
    )

    # Replace empty strings in character and factor columns with NA
    # dt <- replace_empty_with_na(dt, to_view_checks)
  }
}


#### Find PDx

In [200]:
if (!to_chunk) {
  if (to_profvis) {
    p <- profvis({
      # dt <- apply_find_pdx(dt)
      pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
      dt$pdx <- pdx_result$pdx
      dt$pdx_code <- pdx_result$pdx_code
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "find_pdx.html"),
      selfcontained = TRUE
    )
  } else {
    # dt <- apply_find_pdx(dt)
    pdx_result <- apply_find_pdx(dt$clin_c1, dt$clin_c2, dt$clin_icd, acc_pdx)
    dt$pdx <- pdx_result$pdx
    dt$pdx_code <- pdx_result$pdx_code
  }
}


In [201]:
if (to_write) {
  fwrite(dt, here(path_to_intermediate, paste0(
    "output_", year_to_load,
    suffix, ".csv"
  )))
}


## Export

### Export for Batch Grouper

In [202]:
if (to_group) {
  if (to_profvis) {
    p <- profvis({
      export_for_batch_grouper(dt, year_to_load, output_txt_file)
      for_batch_grouping <- fread(output_txt_file,
        sep = "|", na.strings = "--"
      )
      batch_grouping_result <- fread(grouper_result_file,
        sep = "|", na.strings = "--"
      )
    })
    htmlwidgets::saveWidget(
      p,
      file = here("git-ignored-files", "profvis", "export_for_grouper.html"),
      selfcontained = TRUE
    )
  } else {
    export_for_batch_grouper(dt, year_to_load, output_txt_file)
    for_batch_grouping <- fread(output_txt_file,
      sep = "|", na.strings = "--"
    )
    batch_grouping_result <- fread(grouper_result_file,
      sep = "|", na.strings = "--"
    )
  }
}


## Runtime Estimation

### Stop Timer

In [203]:
# Stop the timer and capture total time
toc_data <- toc(log = TRUE)
total_time <- toc_data$toc - toc_data$tic


Total execution time:: 10.19 sec elapsed


### Calculate Speed

In [204]:
# Calculate time spent per cell and per row
total_rows_dt <- nrow(dt)
total_cells <- nrow(dt) * ncol(dt)

time_per_cell <- total_time / total_cells
time_per_row <- total_time / total_rows_dt
time_estimate_total_rows <- time_per_row * total_rows

# Format the row numbers
formatted_total_rows_dt <- format_large_numbers(total_rows_dt)
formatted_total_rows <- format_large_numbers(total_rows)

# Print the results with aligned decimal points and formatted row numbers
cat(sprintf(
  "Time spent (total) for %2s rows:  %1.2f sec  (actual)\n",
  formatted_total_rows_dt, total_time
))
cat(sprintf(
  "Time spent (t/row) for %2s rows:  %1.2f msec (actual)\n",
  formatted_total_rows_dt, time_per_row * 1000
))
cat(sprintf(
  "Time spent (total) for  %2s rows: %2.2f min  (estimate)\n",
  formatted_total_rows, time_estimate_total_rows / 60
))


Time spent (total) for 25.0k rows:  10.19 sec  (actual)
Time spent (t/row) for 25.0k rows:  0.41 msec (actual)
Time spent (total) for  11.8m rows: 80.01 min  (estimate)
